# NASA C-MAPSS preprocessing walkthrough
This notebook is a concise demonstration of the practical 25–30% milestone. The full reusable implementation is in `src/cmapss_pipeline.py`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.cmapss_pipeline import read_cmapss, attach_train_rul, read_official_test_rul, attach_test_rul

## 1. Raw whitespace-separated source

In [ ]:
raw_dir = PROJECT_ROOT / 'data' / 'raw' / 'files'
print((raw_dir / 'train_FD001.txt').read_text().splitlines()[0])

## 2. Parse, name columns, and inspect quality

In [ ]:
train = read_cmapss(raw_dir / 'train_FD001.txt')
display(train.head())
display(pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'missing': train.isna().sum(),
    'unique': train.nunique(),
}).head(10))
print('Exact duplicate rows:', train.duplicated().sum())
print('Duplicate engine-cycle keys:', train.duplicated(['unit_id', 'cycle']).sum())

## 3. Construct training and test RUL labels

In [ ]:
train_labeled = attach_train_rul(train)
display(train_labeled.loc[train_labeled.unit_id == 1, ['unit_id', 'cycle', 'RUL']].tail())
test = read_cmapss(raw_dir / 'test_FD001.txt')
official = read_official_test_rul(raw_dir / 'RUL_FD001.txt')
test_labeled = attach_test_rul(test, official)
display(test_labeled[test_labeled.is_last_observation == 1][['unit_id', 'cycle', 'RUL']].head())

## 4. Review generated evidence and model-ready data

In [ ]:
quality = pd.read_csv(PROJECT_ROOT / 'reports' / 'data_quality_summary.csv')
features = pd.read_csv(PROJECT_ROOT / 'reports' / 'feature_quality_by_subset.csv')
model_ready = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'by_subset' / 'FD001' / 'FD001_train_model_ready.csv', nrows=10)
display(quality)
display(features[(features.dataset_id == 'FD001') & (features.decision != 'retain')])
display(model_ready)

## Interpretation boundary
The files contain simulated degradation trajectories. They are appropriate for ML benchmarking and education, but do not establish maintenance decisions for real aircraft.